# ZGLMM159 — C1 — Valoração e controle de preço — Análise Exploratória

**View:** `dev_procurement.corp_curated.vw_zglmm159_c1`  
**Transação SAP:** ZGLMM159  
**Objetivo:** compreender volumetria, granularidade, chaves, preenchimento, domínios e extremos antes da comparação com o SAP.  
**Colunas no DESCRIBE recebido:** 9

## Como usar
Execute **Run All** no Databricks. As consultas leem a view diretamente e não criam tabelas ou views temporárias.

## Atenções iniciais
- A chave abaixo é apenas uma **candidata** e será confirmada pelos testes de unicidade.
- Colunas comentadas como sem fonte ou `NULL` serão medidas, não presumidas.
- Volumetria igual não comprova qualidade dos dados.


## 1. Metadados e definição da view


In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.vw_zglmm159_c1;


In [0]:
SHOW CREATE TABLE dev_procurement.corp_curated.vw_zglmm159_c1;


## 2. Volumetria geral


In [0]:
SELECT
  COUNT(*) AS total_linhas,
  COUNT(DISTINCT cod_material) AS materiais_distintos,
  COUNT(DISTINCT cod_centro) AS centros_distintos,
  COUNT(DISTINCT cod_empresa) AS empresas_distintas
  
FROM dev_procurement.corp_curated.vw_zglmm159_c1;


## 3. Distribuição por centro e empresa — base para definição de cenários


In [0]:
SELECT
  cod_centro,
  cod_empresa,
  COUNT(*) AS linhas,
  COUNT(DISTINCT cod_material) AS materiais,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_linhas
FROM dev_procurement.corp_curated.vw_zglmm159_c1
GROUP BY cod_centro, cod_empresa
ORDER BY linhas DESC, cod_centro, cod_empresa;


## 4. Granularidade e testes de chave candidata


In [0]:
SELECT 'cod_material' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'))) AS distintos,
ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'))), 0), 6) AS linhas_por_chave
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'))) AS distintos,
ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'))), 0), 6) AS linhas_por_chave
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'cod_material + cod_centro + cod_empresa' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_empresa` AS STRING)), '<NULL>'))) AS distintos,
ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_empresa` AS STRING)), '<NULL>'))), 0), 6) AS linhas_por_chave
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'cod_material + cod_centro + cod_empresa + cod_classe_avaliacao' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_empresa` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_classe_avaliacao` AS STRING)), '<NULL>'))) AS distintos,
ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_empresa` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_classe_avaliacao` AS STRING)), '<NULL>'))), 0), 6) AS linhas_por_chave
FROM dev_procurement.corp_curated.vw_zglmm159_c1;


### Duplicidades na chave candidata: `cod_material + cod_centro + cod_empresa + cod_classe_avaliacao`


In [0]:
SELECT `cod_material`, `cod_centro`, `cod_empresa`, `cod_classe_avaliacao`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.vw_zglmm159_c1
GROUP BY `cod_material`, `cod_centro`, `cod_empresa`, `cod_classe_avaliacao`
HAVING COUNT(*) > 1
ORDER BY qtd DESC, `cod_material`, `cod_centro`, `cod_empresa`, `cod_classe_avaliacao`;


### Inspeção campo a campo das chaves repetidas


In [0]:
WITH duplicadas AS (
  SELECT `cod_material`, `cod_centro`, `cod_empresa`, `cod_classe_avaliacao`
  FROM dev_procurement.corp_curated.vw_zglmm159_c1
  GROUP BY `cod_material`, `cod_centro`, `cod_empresa`, `cod_classe_avaliacao`
  HAVING COUNT(*) > 1
)
SELECT t.*
FROM dev_procurement.corp_curated.vw_zglmm159_c1 t
INNER JOIN duplicadas d ON COALESCE(CAST(t.`cod_material` AS STRING), '<NULL>') = COALESCE(CAST(d.`cod_material` AS STRING), '<NULL>') AND COALESCE(CAST(t.`cod_centro` AS STRING), '<NULL>') = COALESCE(CAST(d.`cod_centro` AS STRING), '<NULL>') AND COALESCE(CAST(t.`cod_empresa` AS STRING), '<NULL>') = COALESCE(CAST(d.`cod_empresa` AS STRING), '<NULL>') AND COALESCE(CAST(t.`cod_classe_avaliacao` AS STRING), '<NULL>') = COALESCE(CAST(d.`cod_classe_avaliacao` AS STRING), '<NULL>')
ORDER BY t.`cod_material`, t.`cod_centro`, t.`cod_empresa`, t.`cod_classe_avaliacao`;


## 5. Preenchimento de todas as colunas — nulo, zero e útil


In [0]:
SELECT 'cod_material' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_material` IS NULL OR LOWER(TRIM(CAST(`cod_material` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_material` IS NULL OR LOWER(TRIM(CAST(`cod_material` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_material` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_material` IS NULL OR LOWER(TRIM(CAST(`cod_material` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_material` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_material` IS NULL OR LOWER(TRIM(CAST(`cod_material` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_material` AS STRING)) AS distintos
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'cod_centro' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_centro` IS NULL OR LOWER(TRIM(CAST(`cod_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_centro` IS NULL OR LOWER(TRIM(CAST(`cod_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_centro` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_centro` IS NULL OR LOWER(TRIM(CAST(`cod_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_centro` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_centro` IS NULL OR LOWER(TRIM(CAST(`cod_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_centro` AS STRING)) AS distintos
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'cod_empresa' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_empresa` IS NULL OR LOWER(TRIM(CAST(`cod_empresa` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_empresa` IS NULL OR LOWER(TRIM(CAST(`cod_empresa` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_empresa` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_empresa` IS NULL OR LOWER(TRIM(CAST(`cod_empresa` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_empresa` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_empresa` IS NULL OR LOWER(TRIM(CAST(`cod_empresa` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_empresa` AS STRING)) AS distintos
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'cod_classe_avaliacao' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_classe_avaliacao` IS NULL OR LOWER(TRIM(CAST(`cod_classe_avaliacao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_classe_avaliacao` IS NULL OR LOWER(TRIM(CAST(`cod_classe_avaliacao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_classe_avaliacao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_classe_avaliacao` IS NULL OR LOWER(TRIM(CAST(`cod_classe_avaliacao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_classe_avaliacao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_classe_avaliacao` IS NULL OR LOWER(TRIM(CAST(`cod_classe_avaliacao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_classe_avaliacao` AS STRING)) AS distintos
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'tp_determinacao_preco' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `tp_determinacao_preco` IS NULL OR LOWER(TRIM(CAST(`tp_determinacao_preco` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`tp_determinacao_preco` IS NULL OR LOWER(TRIM(CAST(`tp_determinacao_preco` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`tp_determinacao_preco` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`tp_determinacao_preco` IS NULL OR LOWER(TRIM(CAST(`tp_determinacao_preco` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`tp_determinacao_preco` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `tp_determinacao_preco` IS NULL OR LOWER(TRIM(CAST(`tp_determinacao_preco` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`tp_determinacao_preco` AS STRING)) AS distintos
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'ind_controle_preco' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `ind_controle_preco` IS NULL OR LOWER(TRIM(CAST(`ind_controle_preco` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`ind_controle_preco` IS NULL OR LOWER(TRIM(CAST(`ind_controle_preco` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`ind_controle_preco` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`ind_controle_preco` IS NULL OR LOWER(TRIM(CAST(`ind_controle_preco` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`ind_controle_preco` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `ind_controle_preco` IS NULL OR LOWER(TRIM(CAST(`ind_controle_preco` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`ind_controle_preco` AS STRING)) AS distintos
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'qt_unidade_preco' AS coluna, 'decimal(5,0)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_unidade_preco` IS NULL OR LOWER(TRIM(CAST(`qt_unidade_preco` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_unidade_preco` IS NULL OR LOWER(TRIM(CAST(`qt_unidade_preco` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_unidade_preco` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_unidade_preco` IS NULL OR LOWER(TRIM(CAST(`qt_unidade_preco` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_unidade_preco` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_unidade_preco` IS NULL OR LOWER(TRIM(CAST(`qt_unidade_preco` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_unidade_preco` AS STRING)) AS distintos
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'vl_preco_medio_movel' AS coluna, 'decimal(18,2)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `vl_preco_medio_movel` IS NULL OR LOWER(TRIM(CAST(`vl_preco_medio_movel` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`vl_preco_medio_movel` IS NULL OR LOWER(TRIM(CAST(`vl_preco_medio_movel` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`vl_preco_medio_movel` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`vl_preco_medio_movel` IS NULL OR LOWER(TRIM(CAST(`vl_preco_medio_movel` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`vl_preco_medio_movel` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `vl_preco_medio_movel` IS NULL OR LOWER(TRIM(CAST(`vl_preco_medio_movel` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`vl_preco_medio_movel` AS STRING)) AS distintos
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'ind_marcado_exclusao' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `ind_marcado_exclusao` IS NULL OR LOWER(TRIM(CAST(`ind_marcado_exclusao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`ind_marcado_exclusao` IS NULL OR LOWER(TRIM(CAST(`ind_marcado_exclusao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`ind_marcado_exclusao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`ind_marcado_exclusao` IS NULL OR LOWER(TRIM(CAST(`ind_marcado_exclusao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`ind_marcado_exclusao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `ind_marcado_exclusao` IS NULL OR LOWER(TRIM(CAST(`ind_marcado_exclusao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`ind_marcado_exclusao` AS STRING)) AS distintos
FROM dev_procurement.corp_curated.vw_zglmm159_c1
ORDER BY pct_nulos DESC, coluna;


## 6. Colunas documentadas como sem fonte ou stub


O DESCRIBE recebido não identifica colunas explicitamente documentadas como sem fonte.


## 7. Cardinalidade e valores dominantes


In [0]:
SELECT 'cod_material' AS coluna, COUNT(DISTINCT CAST(`cod_material` AS STRING)) AS distintos FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'cod_centro' AS coluna, COUNT(DISTINCT CAST(`cod_centro` AS STRING)) AS distintos FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'cod_empresa' AS coluna, COUNT(DISTINCT CAST(`cod_empresa` AS STRING)) AS distintos FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'cod_classe_avaliacao' AS coluna, COUNT(DISTINCT CAST(`cod_classe_avaliacao` AS STRING)) AS distintos FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'tp_determinacao_preco' AS coluna, COUNT(DISTINCT CAST(`tp_determinacao_preco` AS STRING)) AS distintos FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'ind_controle_preco' AS coluna, COUNT(DISTINCT CAST(`ind_controle_preco` AS STRING)) AS distintos FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'qt_unidade_preco' AS coluna, COUNT(DISTINCT CAST(`qt_unidade_preco` AS STRING)) AS distintos FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'vl_preco_medio_movel' AS coluna, COUNT(DISTINCT CAST(`vl_preco_medio_movel` AS STRING)) AS distintos FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'ind_marcado_exclusao' AS coluna, COUNT(DISTINCT CAST(`ind_marcado_exclusao` AS STRING)) AS distintos FROM dev_procurement.corp_curated.vw_zglmm159_c1
ORDER BY distintos ASC, coluna;


In [0]:
-- Domínio: cod_material
SELECT CAST(`cod_material` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM dev_procurement.corp_curated.vw_zglmm159_c1
GROUP BY CAST(`cod_material` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_centro
SELECT CAST(`cod_centro` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM dev_procurement.corp_curated.vw_zglmm159_c1
GROUP BY CAST(`cod_centro` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_empresa
SELECT CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM dev_procurement.corp_curated.vw_zglmm159_c1
GROUP BY CAST(`cod_empresa` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_classe_avaliacao
SELECT CAST(`cod_classe_avaliacao` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM dev_procurement.corp_curated.vw_zglmm159_c1
GROUP BY CAST(`cod_classe_avaliacao` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: tp_determinacao_preco
SELECT CAST(`tp_determinacao_preco` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM dev_procurement.corp_curated.vw_zglmm159_c1
GROUP BY CAST(`tp_determinacao_preco` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: ind_controle_preco
SELECT CAST(`ind_controle_preco` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM dev_procurement.corp_curated.vw_zglmm159_c1
GROUP BY CAST(`ind_controle_preco` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: ind_marcado_exclusao
SELECT CAST(`ind_marcado_exclusao` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM dev_procurement.corp_curated.vw_zglmm159_c1
GROUP BY CAST(`ind_marcado_exclusao` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


## 8. Perfil numérico e extremos


In [0]:
SELECT 'qt_unidade_preco' AS coluna,
MIN(`qt_unidade_preco`) AS minimo,
MAX(`qt_unidade_preco`) AS maximo,
AVG(`qt_unidade_preco`) AS media,
SUM(`qt_unidade_preco`) AS soma,
SUM(CASE WHEN `qt_unidade_preco` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_unidade_preco` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_unidade_preco` > 0 THEN 1 ELSE 0 END) AS positivos
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'vl_preco_medio_movel' AS coluna,
MIN(`vl_preco_medio_movel`) AS minimo,
MAX(`vl_preco_medio_movel`) AS maximo,
AVG(`vl_preco_medio_movel`) AS media,
SUM(`vl_preco_medio_movel`) AS soma,
SUM(CASE WHEN `vl_preco_medio_movel` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `vl_preco_medio_movel` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `vl_preco_medio_movel` > 0 THEN 1 ELSE 0 END) AS positivos
FROM dev_procurement.corp_curated.vw_zglmm159_c1;


## 9. Datas, formatos e faixas


Não há campos com prefixo `dt_` no DESCRIBE recebido.


## 10. Zeros à esquerda e formato das chaves


In [0]:
SELECT 'cod_material' AS coluna,
MIN(LENGTH(TRIM(CAST(`cod_material` AS STRING)))) AS tamanho_min,
MAX(LENGTH(TRIM(CAST(`cod_material` AS STRING)))) AS tamanho_max,
SUM(CASE WHEN TRIM(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]+' THEN 1 ELSE 0 END) AS com_zero_esquerda,
COUNT(DISTINCT TRIM(CAST(`cod_material` AS STRING))) AS distintos_brutos,
COUNT(DISTINCT REGEXP_REPLACE(TRIM(CAST(`cod_material` AS STRING)), '^0+', '')) AS distintos_sem_zero
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'cod_centro' AS coluna,
MIN(LENGTH(TRIM(CAST(`cod_centro` AS STRING)))) AS tamanho_min,
MAX(LENGTH(TRIM(CAST(`cod_centro` AS STRING)))) AS tamanho_max,
SUM(CASE WHEN TRIM(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]+' THEN 1 ELSE 0 END) AS com_zero_esquerda,
COUNT(DISTINCT TRIM(CAST(`cod_centro` AS STRING))) AS distintos_brutos,
COUNT(DISTINCT REGEXP_REPLACE(TRIM(CAST(`cod_centro` AS STRING)), '^0+', '')) AS distintos_sem_zero
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'cod_empresa' AS coluna,
MIN(LENGTH(TRIM(CAST(`cod_empresa` AS STRING)))) AS tamanho_min,
MAX(LENGTH(TRIM(CAST(`cod_empresa` AS STRING)))) AS tamanho_max,
SUM(CASE WHEN TRIM(CAST(`cod_empresa` AS STRING)) RLIKE '^0[0-9]+' THEN 1 ELSE 0 END) AS com_zero_esquerda,
COUNT(DISTINCT TRIM(CAST(`cod_empresa` AS STRING))) AS distintos_brutos,
COUNT(DISTINCT REGEXP_REPLACE(TRIM(CAST(`cod_empresa` AS STRING)), '^0+', '')) AS distintos_sem_zero
FROM dev_procurement.corp_curated.vw_zglmm159_c1
UNION ALL
SELECT 'cod_classe_avaliacao' AS coluna,
MIN(LENGTH(TRIM(CAST(`cod_classe_avaliacao` AS STRING)))) AS tamanho_min,
MAX(LENGTH(TRIM(CAST(`cod_classe_avaliacao` AS STRING)))) AS tamanho_max,
SUM(CASE WHEN TRIM(CAST(`cod_classe_avaliacao` AS STRING)) RLIKE '^0[0-9]+' THEN 1 ELSE 0 END) AS com_zero_esquerda,
COUNT(DISTINCT TRIM(CAST(`cod_classe_avaliacao` AS STRING))) AS distintos_brutos,
COUNT(DISTINCT REGEXP_REPLACE(TRIM(CAST(`cod_classe_avaliacao` AS STRING)), '^0+', '')) AS distintos_sem_zero
FROM dev_procurement.corp_curated.vw_zglmm159_c1;


## 11. Linhas integralmente duplicadas


In [0]:
SELECT
  COUNT(*) AS linhas,
  COUNT(DISTINCT SHA2(CONCAT_WS('§', COALESCE(CAST(`cod_material` AS STRING), '<NULL>'), COALESCE(CAST(`cod_centro` AS STRING), '<NULL>'), COALESCE(CAST(`cod_empresa` AS STRING), '<NULL>'), COALESCE(CAST(`cod_classe_avaliacao` AS STRING), '<NULL>'), COALESCE(CAST(`tp_determinacao_preco` AS STRING), '<NULL>'), COALESCE(CAST(`ind_controle_preco` AS STRING), '<NULL>'), COALESCE(CAST(`qt_unidade_preco` AS STRING), '<NULL>'), COALESCE(CAST(`vl_preco_medio_movel` AS STRING), '<NULL>'), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '<NULL>')), 256)) AS linhas_distintas,
  COUNT(*) - COUNT(DISTINCT SHA2(CONCAT_WS('§', COALESCE(CAST(`cod_material` AS STRING), '<NULL>'), COALESCE(CAST(`cod_centro` AS STRING), '<NULL>'), COALESCE(CAST(`cod_empresa` AS STRING), '<NULL>'), COALESCE(CAST(`cod_classe_avaliacao` AS STRING), '<NULL>'), COALESCE(CAST(`tp_determinacao_preco` AS STRING), '<NULL>'), COALESCE(CAST(`ind_controle_preco` AS STRING), '<NULL>'), COALESCE(CAST(`qt_unidade_preco` AS STRING), '<NULL>'), COALESCE(CAST(`vl_preco_medio_movel` AS STRING), '<NULL>'), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '<NULL>')), 256)) AS duplicatas_integrais
FROM dev_procurement.corp_curated.vw_zglmm159_c1;


## 12. Amostra ampliada — validar formatos reais


In [0]:
SELECT * FROM dev_procurement.corp_curated.vw_zglmm159_c1 ORDER BY cod_centro, cod_material LIMIT 100;


## 13. Resumo automático de cenário e chave


In [0]:
WITH base AS (
  SELECT COUNT(*) AS linhas,
         COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_empresa` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_classe_avaliacao` AS STRING)), '<NULL>'))) AS chaves
  FROM dev_procurement.corp_curated.vw_zglmm159_c1
)
SELECT 'CENARIO' AS bloco, 'tabela' AS item, 'dev_procurement.corp_curated.vw_zglmm159_c1' AS valor
UNION ALL SELECT 'CENARIO', 'colunas', '9'
UNION ALL SELECT 'CENARIO', 'linhas', CAST(linhas AS STRING) FROM base
UNION ALL SELECT 'CHAVE CANDIDATA', 'campos', 'cod_material + cod_centro + cod_empresa + cod_classe_avaliacao'
UNION ALL SELECT 'CHAVE CANDIDATA', 'distintos', CAST(chaves AS STRING) FROM base
UNION ALL SELECT 'CHAVE CANDIDATA', 'veredito', CASE WHEN linhas = chaves THEN 'CHAVE UNICA' ELSE 'NAO UNICA - INVESTIGAR DIMENSAO ADICIONAL' END FROM base;


---
## Próximo passo

1. Revisar a granularidade real e a chave candidata.
2. Confirmar se as colunas documentadas como sem fonte permanecem realmente nulas.
3. Escolher um recorte reproduzível para extração SAP, preservando todas as visões C1, M1 e M2.
4. Não avançar para o de/para antes de validar o diagnóstico desta análise exploratória.
